# CSCIE89 HW05 – Problem 2: Batch Normalization in the flower CNN

This notebook starts from **1_Augmentation_CNN_Classify_flowers** and keeps its code. The only additions for Problem 2 are:

- A new network, `BatchNormCNN`: the same `BasicCNN` with a **Batch Normalization layer (`nn.BatchNorm2d`) after each `Conv2d` layer**.
- A switch `USE_BATCH_NORM` (next to `USE_AUGMENTATION`) to choose the network.
- Overall **precision** on the **entire test set** (all 551 test images), saved for every run.
- A final section that compares the four runs and reports the results.

The model size (64 × 64), Adam optimizer, learning rate, epochs, batch size, and data split stay the same in every run.

**Run the notebook four times, top to bottom**, changing only the two switches in section 1. Each run saves its metrics in its own `outputs/` folder; section 11 reads all four.

| Run | `USE_BATCH_NORM` | `USE_AUGMENTATION` |
|---|---|---|
| 1. Original network, no augmentation | `False` | `False` |
| 2. Original network, with augmentation | `False` | `True` |
| 3. BatchNorm network, no augmentation | `True` | `False` |
| 4. BatchNorm network, with augmentation | `True` | `True` |

## 1. Install packages and import tools

If the imports fail, uncomment the installation line, run it once, then restart the kernel. TensorFlow is not required, even for the Flower Photos dataset.

`precision_score` is added to the scikit-learn imports to compute overall precision.

In [ ]:
# %pip install torch torchvision scikit-learn matplotlib pillow   # Uncomment once if the packages are missing.

from pathlib import Path                         # Path builds file paths that work on any OS.
import copy                                      # deepcopy keeps a frozen copy of the best weights.
import json                                      # Save and read metrics as human-readable JSON files.
import random                                    # Python's random-number generator (seeded below).
import time                                      # Measure how long training takes.
import numpy as np                               # Arrays and numeric helpers.
import matplotlib.pyplot as plt                  # Plotting library for images and charts.
from PIL import Image                            # Pillow opens image files.
import torch                                     # PyTorch: tensors and training tools.
from torch import nn                             # Neural-network layers (Conv2d, BatchNorm2d, Linear, ...).
from torch.utils.data import Dataset, DataLoader # Dataset loads one photo; DataLoader groups photos into batches.
from torchvision import datasets, transforms     # ImageFolder dataset reader and image transformations.
from torchvision.datasets.utils import download_and_extract_archive  # Download and unpack the dataset archive.
from sklearn.model_selection import train_test_split  # Reproducible, stratified data split.
from sklearn.metrics import (                    # Evaluation metrics:
    accuracy_score, balanced_accuracy_score, classification_report,  # accuracy, balanced accuracy, per-class report,
    ConfusionMatrixDisplay, precision_score,     # confusion-matrix plot, and overall precision (added for Problem 2).
)

# Seeds make the split and initial weights repeatable, but do not guarantee
# identical floating-point results across all devices or PyTorch versions.
SEED = 42                                        # One seed used everywhere.
random.seed(SEED)                                # Seed Python's random module.
np.random.seed(SEED)                             # Seed NumPy.
torch.manual_seed(SEED)                          # Seed PyTorch (weight initialization, augmentation).
torch.set_num_threads(4)  # Avoid excessive CPU thread overhead on this small model.
device = torch.device('cuda' if torch.cuda.is_available() else          # Use an NVIDIA GPU if present,
                      'mps' if torch.backends.mps.is_available() else 'cpu')  # else Apple GPU, else the CPU.
IMAGE_SIZE = 64                                  # Every photo is resized to 64 x 64 pixels.
BATCH_SIZE = 64                                  # Photos per training step.
EPOCHS = 30                                      # Complete passes through the training data.
USE_AUGMENTATION = True  # False gives the original resize-only baseline.
USE_BATCH_NORM = True    # Problem 2: True adds BatchNorm2d after each Conv2d; False is the original BasicCNN.
LEARNING_RATE = 0.001                            # Adam step size.
DATA_DIR = Path('../data')                       # Folder for the downloaded dataset.
DATA_DIR.mkdir(exist_ok=True)                    # Create it; no error if it already exists.
print('PyTorch:', torch.__version__, '| Device:', device)  # Show the PyTorch version and the device in use.
print('Batch normalization:', USE_BATCH_NORM, '| Augmentation:', USE_AUGMENTATION)  # Show which experiment this run is.

## 2. Download Flower Photos and read the labels

The [Flower Photos dataset](https://www.tensorflow.org/tutorials/images/classification) has 3,670 photographs in five folders: daisy, dandelion, roses, sunflowers, and tulips. The archive is approximately 220 MB.

`ImageFolder` assigns labels from alphabetically sorted folder names (daisy, dandelion, roses, ...). We read that mapping instead of guessing the numbers. This dataset has no official test split, so we will make a reproducible stratified 70%/15%/15% split.

In [ ]:
flower_root = DATA_DIR / 'flower_photos'         # Folder the archive unpacks into.
if not flower_root.exists():                     # Download only once.
    download_and_extract_archive(                # Download the archive and unpack it:
        'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',  # dataset URL,
        download_root=str(DATA_DIR),             # saved inside DATA_DIR.
    )
catalog = datasets.ImageFolder(flower_root)      # Finds every photo; the folder name is the label.
class_names = catalog.classes                    # Class names in label order.
# Each record is (path_to_image, integer_class_label).
records = [(Path(path), label) for path, label in catalog.samples]  # List of all photos with their labels.
labels = [label for _, label in records]         # Just the labels, used to stratify the split.
train_records, remaining = train_test_split(     # First split: 70% training, 30% set aside,
    records, test_size=0.30, random_state=SEED, stratify=labels)  # keeping class proportions equal.
val_records, test_records = train_test_split(    # Second split: the 30% becomes 15% validation + 15% test,
    remaining, test_size=0.50, random_state=SEED,
    stratify=[label for _, label in remaining])  # again keeping class proportions equal.
print('Folder-to-label mapping:', catalog.class_to_idx)  # Show which number each class received.

## 3. Check the split and look at real images

Training images teach the weights. Validation images select the best epoch. Test images are reserved for final evaluation, after those decisions. Never adjust settings based on test results; use validation results for experiments.

In [ ]:
# A path must appear in only one split: this checks for accidental overlap.
train_paths = {path for path, _ in train_records}   # Set of training file paths.
val_paths = {path for path, _ in val_records}       # Set of validation file paths.
test_paths = {path for path, _ in test_records}     # Set of test file paths.
assert train_paths.isdisjoint(val_paths)            # No photo is in both training and validation.
assert train_paths.isdisjoint(test_paths)           # No photo is in both training and test.
assert val_paths.isdisjoint(test_paths)             # No photo is in both validation and test.
for name, records in [('Train', train_records), ('Validation', val_records), ('Test', test_records)]:  # For each split:
    counts = np.bincount([label for _, label in records], minlength=len(class_names))  # count photos per class
    print(f'{name}: {len(records)} images | {dict(zip(class_names, counts.tolist()))}')  # and print the counts.

# Show two training examples per class, before resizing.
fig, axes = plt.subplots(2, len(class_names), figsize=(3 * len(class_names), 5), squeeze=False)  # 2 rows x 5 classes grid.
for label, name in enumerate(class_names):          # One column per class.
    examples = [path for path, target in train_records if target == label][:2]  # First two training photos of this class.
    for row, path in enumerate(examples):           # One row per example.
        with Image.open(path) as picture:           # Open the photo file.
            axes[row, label].imshow(picture.convert('RGB'))  # Draw it in its panel.
        axes[row, label].set_title(name)            # Class name as the title.
        axes[row, label].axis('off')                # Hide axis ticks.
plt.tight_layout()                                  # Reduce overlap between panels.
plt.show()                                          # Display the figure.

## 4. Add variation to training images only

We use three gentle changes that preserve the flower label:

- **Horizontal flip:** mirror the photo left-to-right with a 50% chance.
- **Rotation:** turn the photo by a random angle between −15° and +15°.
- **Color jitter:** slightly change brightness and contrast to mimic lighting variation. We do not change hue because flower color can help classification.

Each time a training photo is loaded, new random changes are sampled. No extra files are created, and the number of training images stays the same.

All photos are resized to 64 × 64. `ToTensor()` converts RGB pixels to a tensor with shape `[3, 64, 64]` and values between 0 and 1.

**Validation, test, and new prediction images get only resizing and tensor conversion.** Their preprocessing must stay consistent so the scores are meaningful. Augmentation is a training aid, not part of evaluating the model.

`PhotoDataset` loads one photo at a time; `DataLoader` groups photos into batches.

In [ ]:
# Deterministic preparation for validation, test, and single-image prediction.
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # Resize to 64 x 64.
    transforms.ToTensor(),                          # Convert to a [3, 64, 64] tensor with values 0..1.
])

# Random changes are applied ONLY to the training images.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # Resize to 64 x 64.
    transforms.RandomHorizontalFlip(p=0.5),  # Mirror approximately half the photos.
    transforms.RandomRotation(15),          # Small rotations; corners may be padded black.
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Gentle lighting variation.
    transforms.ToTensor(),                  # Convert to [channels, height, width].
])
if not USE_AUGMENTATION:                    # Baseline run:
    train_transform = image_transform      # Same preprocessing as the original baseline.

class PhotoDataset(Dataset):                # Loads photos from a list of (path, label) records.
    def __init__(self, records, transform):
        self.records = records              # Remember the (path, label) list.
        self.transform = transform          # Remember the preprocessing to apply.

    def __len__(self):
        return len(self.records)            # Number of photos in this dataset.

    def __getitem__(self, index):
        path, label = self.records[index]   # Look up one record.
        with Image.open(path) as picture:   # Open the photo file.
            # Convert even grayscale/RGBA files to three-channel RGB.
            image = self.transform(picture.convert('RGB'))  # Then apply the transform -> tensor.
        return image, label                 # One (image tensor, label) pair.

train_dataset = PhotoDataset(train_records, train_transform)  # Training photos (augmented if switched on).
val_dataset = PhotoDataset(val_records, image_transform)      # Validation photos: resize only.
test_dataset = PhotoDataset(test_records, image_transform)    # Test photos: resize only.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,       # Batches of 64 training photos,
                          shuffle=True,                               # in a new random order each epoch,
                          generator=torch.Generator().manual_seed(SEED),  # with a repeatable shuffle,
                          num_workers=0)                              # loaded in the main process.

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)    # Validation batches, fixed order.
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)  # Test batches, fixed order; covers ALL test photos.
images, targets = next(iter(train_loader))  # Take one batch to check the shapes.

print('Images:', images.shape, '| Labels:', targets.shape, targets.dtype)  # Expect [64, 3, 64, 64] and [64].
assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)  # Each image is 3 x 64 x 64.

### See what augmentation does

The first image below uses ordinary resizing. The next four are random versions of the **same training photo**. The label is unchanged. Mild changes provide variety without hiding the flower.

**NOTE**

Augmentation does not increase the dataset’s record count in this notebook. It changes images randomly when they are loaded.
- The training set still contains 2,569 records.
- Each epoch processes those same records, but their images may look different because of random flips, rotations, and lighting changes.
- No additional image files are saved.

In [ ]:
# Preview one training image; no validation or test images are augmented.
preview_path, preview_label = train_records[0]          # First training photo and its label.
with Image.open(preview_path) as photo:                 # Open the photo file.
    photo = photo.convert('RGB')                        # Make sure it has three color channels.
    fig, axes = plt.subplots(1, 5, figsize=(12, 3))     # One row of five panels.
    for index, axis in enumerate(axes):                 # For each panel:
        transform = image_transform if index == 0 else train_transform  # first is resize only, others use training transform.
        picture = transform(photo)                      # Apply it; result is a [3, 64, 64] tensor.
        axis.imshow(picture.permute(1, 2, 0))  # Plotting uses height, width, channels.
        axis.set_title('Resize only' if index == 0 else f'Training view {index}')  # Panel title.
        axis.axis('off')                                # Hide axis ticks.
    fig.suptitle(f'Same flower, same label: {class_names[preview_label]}')  # Figure title with the class name.
    plt.tight_layout()                                  # Reduce overlap between panels.
    plt.show()                                          # Display the figure.

## 5. Build the CNNs: the original `BasicCNN` and the new `BatchNormCNN`

A convolution learns small filters that detect patterns such as edges and textures. ReLU replaces negative values with zero and enables nonlinear relationships. Max pooling halves the spatial dimensions, retaining strong responses. Flatten turns the final feature maps into a vector; linear layers turn that vector into class scores, called logits.

For one image the shapes are: `3×64×64 → 16×32×32 → 32×16×16 → 64×8×8 → 4096 → 64 → number of classes`. Each arrow through the feature extractor includes convolution, ReLU, and pooling.

The flower classifier returns five output logits, one per class. Cross-entropy expects integer targets, and internally handles the softmax-related calculation; do not add softmax before this loss.

**Problem 2 – Batch Normalization.** `BatchNormCNN` is `BasicCNN` with one `nn.BatchNorm2d` layer **after each `Conv2d`** (Conv → BatchNorm → ReLU → MaxPool). For every channel, BatchNorm rescales the convolution outputs in a batch to mean 0 and standard deviation 1, then applies a learned scale and shift. This usually makes training faster and more stable. During training it uses the statistics of the current batch; in `model.eval()` mode it uses running averages saved during training, which is why the notebook already switches between `model.train()` and `model.eval()`. The number passed to `BatchNorm2d` is the number of channels coming out of the preceding `Conv2d` (16, 32, 64). Shapes do not change.

In [ ]:
class BasicCNN(nn.Module):                       # Original network from the class notebook (unchanged).
    def __init__(self, num_classes):
        super().__init__()                       # Set up the nn.Module machinery.
        self.features = nn.Sequential(           # Feature extractor: three Conv -> ReLU -> Pool blocks.
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 color channels -> 16 feature maps, 3x3 filters.
            nn.ReLU(),                                    # Keep positives, zero out negatives.
            nn.MaxPool2d(2),                              # Halve height and width: 64 -> 32.
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 feature maps.
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 32 -> 16.
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 -> 64 feature maps.
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 16 -> 8.
        )
        self.classifier = nn.Sequential(         # Classifier: feature maps -> class scores.
            nn.Flatten(),                                 # 64 x 8 x 8 maps -> vector of 4096 numbers.
            nn.Linear(64 * (IMAGE_SIZE // 8) ** 2, 64),   # 4096 -> 64 hidden units.
            nn.ReLU(),
            nn.Linear(64, num_classes),                   # 64 -> one logit per class.
        )

    def forward(self, x):
        return self.classifier(self.features(x)) # Run the features, then the classifier.

class BatchNormCNN(nn.Module):                   # Problem 2: BasicCNN + BatchNorm2d after each Conv2d.
    def __init__(self, num_classes):
        super().__init__()                       # Set up the nn.Module machinery.
        self.features = nn.Sequential(           # Feature extractor: three Conv -> BatchNorm -> ReLU -> Pool blocks.
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 color channels -> 16 feature maps.
            nn.BatchNorm2d(16),                           # NEW: normalize each of the 16 channels.
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 64 -> 32.
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 feature maps.
            nn.BatchNorm2d(32),                           # NEW: normalize each of the 32 channels.
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 32 -> 16.
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 -> 64 feature maps.
            nn.BatchNorm2d(64),                           # NEW: normalize each of the 64 channels.
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 16 -> 8.
        )
        self.classifier = nn.Sequential(         # Same classifier as BasicCNN.
            nn.Flatten(),                                 # 64 x 8 x 8 -> 4096.
            nn.Linear(64 * (IMAGE_SIZE // 8) ** 2, 64),   # 4096 -> 64.
            nn.ReLU(),
            nn.Linear(64, num_classes),                   # 64 -> one logit per class.
        )

    def forward(self, x):
        return self.classifier(self.features(x)) # Run the features, then the classifier.

CNN = BatchNormCNN if USE_BATCH_NORM else BasicCNN  # Pick the network for this run.

# Reset before model creation so both experiments start with the same weights.
torch.manual_seed(SEED)
model = CNN(len(class_names)).to(device)         # Build the chosen network and move it to the device.
criterion = nn.CrossEntropyLoss()                # Loss for multi-class classification.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Adam updates all weights.
print(model)                                     # Show the layers.
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))  # Count the weights.
with torch.no_grad():                            # Quick check without building a gradient graph:
    assert model(images.to(device)).shape == (len(images), len(class_names))  # one row of 5 logits per image.

## 6. Define evaluation, then train

One epoch is a complete pass through training data. Each batch follows five steps: clear gradients, predict, calculate loss, backpropagate, update weights. Evaluation uses `model.eval()` and `torch.no_grad()` because it must not update weights or build a gradient graph. (For `BatchNormCNN`, `model.eval()` also makes BatchNorm use its saved running statistics instead of the current batch.)

We multiply each batch's mean loss by its actual size before averaging across examples, so a smaller final batch has the correct weight. Training metrics below are measured while weights change; validation metrics use the fixed weights at the end of each epoch.

We keep the weights with the lowest **validation loss**, not the highest test accuracy. The test loader is not used during training.

Training accuracy is measured on randomly changed images while weights are being updated. Validation accuracy uses unchanged images and fixed weights. Augmentation can make training accuracy lower even when generalization improves.

In [ ]:
def evaluate(model, loader):                     # Average loss and accuracy over a whole loader.
    model.eval()                                 # Evaluation mode (BatchNorm uses running statistics).
    loss_sum, correct, count = 0.0, 0, 0         # Running totals.
    with torch.no_grad():                        # No gradients needed.
        for inputs, labels in loader:            # For each batch:
            inputs, labels = inputs.to(device), labels.to(device)  # move it to the device,
            logits = model(inputs)               # predict class scores,
            loss_sum += criterion(logits, labels).item() * len(labels)  # add the batch's total loss,
            correct += (logits.argmax(1) == labels).sum().item()   # count correct predictions,
            count += len(labels)                 # and count the images.
    return loss_sum / count, correct / count     # Mean loss and accuracy.

# Reset random choices and batch order after the preview cells.
torch.manual_seed(SEED)
train_loader.generator.manual_seed(SEED)

history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}  # Metrics per epoch.
best_val_loss = float('inf')                     # Lowest validation loss so far.
best_state = None                                # Weights at that epoch.
start = time.perf_counter()                      # Start the timer.
for epoch in range(1, EPOCHS + 1):               # Repeat for every epoch:
    model.train()                                # Training mode (BatchNorm uses batch statistics).
    loss_sum, correct, count = 0.0, 0, 0         # Running totals for this epoch.
    for inputs, labels in train_loader:          # For each training batch:
        inputs, labels = inputs.to(device), labels.to(device)  # Move the batch to the device.
        optimizer.zero_grad()             # Clear gradients from the last batch.
        logits = model(inputs)            # Forward pass.
        loss = criterion(logits, labels)  # Compare scores with true labels.
        loss.backward()                   # Calculate weight gradients.
        optimizer.step()                  # Update weights using Adam.
        loss_sum += loss.item() * len(labels)                  # Add the batch's total loss.
        correct += (logits.argmax(1) == labels).sum().item()   # Count correct predictions.
        count += len(labels)                                   # Count the images.
    train_loss, train_accuracy = loss_sum / count, correct / count  # Training metrics for this epoch.
    val_loss, val_accuracy = evaluate(model, val_loader)       # Validation metrics with fixed weights.
    for key, value in [('train_loss', train_loss), ('val_loss', val_loss),
                       ('train_accuracy', train_accuracy), ('val_accuracy', val_accuracy)]:
        history[key].append(value)               # Record all four numbers.
    if val_loss < best_val_loss:                 # New best epoch on validation loss:
        best_val_loss, best_epoch = val_loss, epoch  # remember it,
        # A copy is necessary: a plain reference would keep changing during training.
        best_state = copy.deepcopy(model.state_dict())  # and keep a copy of its weights.
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.3f}, acc {train_accuracy:.1%}'
          f' | val loss {val_loss:.3f}, acc {val_accuracy:.1%}', flush=True)  # Progress line.

model.load_state_dict(best_state)                # Go back to the best epoch's weights.
print(f'Restored epoch {best_epoch}, selected using validation loss.')
print(f'Training time: {(time.perf_counter() - start) / 60:.1f} minutes')

## 7. Plot learning curves

Lower loss and higher accuracy generally indicate learning. Training improvement accompanied by worsening validation loss suggests overfitting: the model fits training details that do not generalize. The dashed line marks the selected epoch.

In [ ]:
epochs = np.arange(1, EPOCHS + 1)                # x-axis: 1, 2, ..., EPOCHS.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Two plots side by side: loss and accuracy.
for ax, metric in zip(axes, ['loss', 'accuracy']):  # For each metric:
    ax.plot(epochs, history['train_' + metric], label='Training')    # training curve,
    ax.plot(epochs, history['val_' + metric], label='Validation')    # validation curve,
    ax.axvline(best_epoch, color='gray', linestyle='--', label='Selected epoch')  # selected epoch,
    ax.set(xlabel='Epoch', ylabel=metric.capitalize(), title=metric.capitalize() + ' by epoch')  # labels,
    ax.legend()                                  # legend,
    ax.grid(alpha=0.25)                          # light grid.
axes[1].set_ylim(0, 1)                           # Accuracy axis from 0 to 1.
plt.tight_layout()                               # Reduce overlap.
plt.show()                                       # Display the figure.

## 8. Evaluate the selected model on the test set

**Accuracy** is the fraction correct. **Precision** asks how often predictions of a class are right; **recall** asks how many real members of that class we found. **F1** balances precision and recall. **Support** is the number of examples in a class. Macro averages give each class equal weight; weighted averages give more weight to larger classes.

Balanced accuracy averages recall across classes, which helps when class counts differ (the five flower classes have different numbers of photos). We also compare with a trivial model that always predicts the most common *training* class. Confusion-matrix rows are true classes and columns are predictions.

**Problem 2 – overall precision on the entire test set.** `test_loader` goes through **all 551 test images** (the `assert` below checks this). We compute overall precision two ways with `precision_score`: **macro** (the average of the five per-class precisions; every class counts equally) and **weighted** (each class weighted by its number of test images). Macro precision is the main number we compare.

In [ ]:
model.eval()                                     # Evaluation mode (BatchNorm uses running statistics).
true_labels, predicted_labels, probabilities = [], [], []  # Collected over the whole test set.
test_loss_sum = 0.0                              # Running total of test loss.
with torch.no_grad():                            # No gradients needed.
    for inputs, labels in test_loader:           # Every test batch, so every test image:
        logits = model(inputs.to(device))        # predict class scores,
        test_loss_sum += criterion(logits, labels.to(device)).item() * len(labels)  # add the batch's total loss.
        # Softmax is useful here to display scores as probabilities.
        # These are model scores, not guarantees or calibrated confidence estimates.
        probabilities.extend(logits.softmax(1).cpu().numpy())    # Keep the probabilities,
        predicted_labels.extend(logits.argmax(1).cpu().numpy())  # the predicted classes,
        true_labels.extend(labels.numpy())                       # and the true classes.
true_labels = np.asarray(true_labels)            # Lists -> NumPy arrays.
predicted_labels = np.asarray(predicted_labels)
probabilities = np.asarray(probabilities)
assert len(true_labels) == len(test_dataset)     # Problem 2: confirm the ENTIRE test set was evaluated.
test_loss = test_loss_sum / len(true_labels)     # Mean test loss.
test_accuracy = accuracy_score(true_labels, predicted_labels)            # Fraction correct.
balanced_accuracy = balanced_accuracy_score(true_labels, predicted_labels)  # Mean per-class recall.
precision_macro = precision_score(true_labels, predicted_labels, average='macro', zero_division=0)  # Problem 2: overall precision, classes equal.
precision_weighted = precision_score(true_labels, predicted_labels, average='weighted', zero_division=0)  # Problem 2: overall precision, weighted by class size.
precision_per_class = precision_score(true_labels, predicted_labels, average=None,
                                      labels=list(range(len(class_names))), zero_division=0)  # Problem 2: precision of each class.
majority_class = np.bincount([label for _, label in train_records]).argmax()  # Most common training class.
baseline_accuracy = np.mean(true_labels == majority_class)  # Accuracy of always predicting it.
print(f'Test images evaluated: {len(true_labels)} of {len(test_dataset)}')  # Problem 2: show the full test set was used.
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2%}')
print(f'Balanced accuracy: {balanced_accuracy:.2%}')
print(f'Overall precision (macro): {precision_macro:.2%}')        # Problem 2.
print(f'Overall precision (weighted): {precision_weighted:.2%}')  # Problem 2.
print(f'Always predict {class_names[majority_class]} baseline: {baseline_accuracy:.2%}')
print(classification_report(true_labels, predicted_labels,       # Per-class precision, recall, F1, support.
      labels=list(range(len(class_names))), target_names=class_names, zero_division=0))
fig, ax = plt.subplots(figsize=(7, 6))           # Figure for the confusion matrix.
ConfusionMatrixDisplay.from_predictions(true_labels, predicted_labels,  # Rows = true class, columns = predicted.
    labels=list(range(len(class_names))), display_labels=class_names,
    cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Test confusion matrix')
plt.tight_layout()
plt.show()

## 9. Inspect predictions and mistakes

Visual inspection helps reveal ambiguous photos and recurring errors. The first grid is a reproducible random test sample; the second contains only mistakes. Correct titles are green and incorrect titles are red.

In [ ]:
def show_predictions(indices, title):            # Show test images with true and predicted labels.
    if len(indices) == 0:                        # Nothing to show:
        print('No mistakes in this test run.')
        return
    fig, axes = plt.subplots((len(indices) + 3) // 4, 4,                    # Grid with 4 columns and
                             figsize=(12, 3 * ((len(indices) + 3) // 4)), squeeze=False)  # as many rows as needed.
    for ax in axes.flat:
        ax.axis('off')                           # Hide ticks (also blanks unused panels).
    for ax, index in zip(axes.flat, indices):    # One panel per chosen test image:
        image, actual = test_dataset[int(index)] # load the image and its true label,
        predicted = predicted_labels[index]      # look up the prediction.
        # Plotting needs H,W,C, whereas PyTorch stores C,H,W.
        ax.imshow(image.permute(1, 2, 0))
        ax.set_title(f'True: {class_names[actual]} | Pred: {class_names[predicted]}\n'
                     f'Score: {probabilities[index, predicted]:.1%}',
                     color='green' if actual == predicted else 'red', fontsize=9)  # Green if right, red if wrong.
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(SEED)                # Repeatable random generator.
show_predictions(rng.choice(len(test_dataset), size=12, replace=False), 'Sample test predictions')  # 12 random test images.
errors = np.flatnonzero(true_labels != predicted_labels)  # Indices of the wrong predictions.
show_predictions(errors[:8], 'Examples the model got wrong')  # First 8 mistakes.

## 10. Save, reload, and predict one image

We save the weights together with class names and image size so the label mapping is not lost. Reconstruct the same model architecture before loading weights. The example below checks that reloaded weights give the same scores and demonstrates prediction for a single image.

For Problem 2 the folder name also records the network (`basic` or `batchnorm`), so all four runs are kept, and the precision numbers are added to `metrics.json`.

In [ ]:
# Keep each network/augmentation combination separate, preserving earlier checkpoints.
run_name = (f"flowers_{'batchnorm' if USE_BATCH_NORM else 'basic'}"            # e.g. flowers_batchnorm_augmented
            f"_{'augmented' if USE_AUGMENTATION else 'baseline'}")
OUTPUT_DIR = Path('outputs') / run_name          # One folder per run.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)    # Create it if needed.
model_path = OUTPUT_DIR / 'basic_cnn.pth'        # Checkpoint file name.
torch.save({                                     # Save weights plus the information needed to reuse them.
    'state_dict': {key: value.cpu() for key, value in model.state_dict().items()},  # Weights (and BatchNorm running stats).
    'class_names': class_names, 'image_size': IMAGE_SIZE, 'best_epoch': best_epoch,
}, model_path)
# Store metrics separately in a human-readable file.
metrics = {'augmentation': USE_AUGMENTATION, 'batch_norm': USE_BATCH_NORM, 'seed': SEED,  # Which experiment this is.
           'selected_val_loss': best_val_loss,
           'selected_val_accuracy': history['val_accuracy'][best_epoch - 1],
           'test_loss': test_loss, 'test_accuracy': test_accuracy,
           'balanced_accuracy': balanced_accuracy, 'majority_baseline_accuracy': float(baseline_accuracy),
           'precision_macro': precision_macro, 'precision_weighted': precision_weighted,  # Problem 2: overall precision.
           'precision_per_class': dict(zip(class_names, precision_per_class.tolist())),  # Problem 2: per-class precision.
           'test_images': len(true_labels),      # Problem 2: number of test images evaluated.
           'best_epoch': best_epoch, 'history': history}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))  # Write the metrics file.

checkpoint = torch.load(model_path, map_location='cpu', weights_only=True)  # Load the checkpoint back.
assert checkpoint['image_size'] == IMAGE_SIZE    # Same image size as now.
reloaded = CNN(len(checkpoint['class_names'])).to(device)  # Rebuild the same network type.
reloaded.load_state_dict(checkpoint['state_dict'])  # Put the saved weights into it.
reloaded.eval()                                  # Evaluation mode.
one_image, actual = test_dataset[0]              # One test image and its label.
with torch.no_grad():
    batch = one_image.unsqueeze(0).to(device)  # Add batch dimension: (1,3,64,64).
    torch.testing.assert_close(model(batch), reloaded(batch))  # Reloaded model gives the same scores.
    predicted = reloaded(batch).argmax(1).item() # Predicted class number.
print('Reload verified. Example:', checkpoint['class_names'][predicted],
      '| Actual:', class_names[actual])
print('Saved to:', model_path.resolve())

def predict_image(path):                         # Predict the class of any photo file.
    with Image.open(path) as image:
        batch = image_transform(image.convert('RGB')).unsqueeze(0).to(device)  # Same preprocessing as test images.
    with torch.no_grad():
        scores = reloaded(batch).softmax(1)[0].cpu()  # Probabilities for the five classes.
    index = scores.argmax().item()               # Most likely class.
    return checkpoint['class_names'][index], scores[index].item()  # Its name and score.

# Replace this with your own photo path to try another image.
print('Single-image prediction:', predict_image(test_records[0][0]))

## 11. Problem 2 – Compare the original and BatchNorm networks, without and with augmentation

This cell reads the `metrics.json` saved by each of the four runs (section 10) and compares them. Runs that have not been done yet are skipped, so run the notebook four times first (see the table at the top). All numbers are on the **entire test set** of 551 images.

In [ ]:
rows = []                                        # One entry per finished run.
for network in ['basic', 'batchnorm']:           # Original network, then BatchNorm network.
    for augmentation in ['baseline', 'augmented']:  # Without, then with augmentation.
        metrics_path = Path('outputs') / f'flowers_{network}_{augmentation}' / 'metrics.json'  # Where that run saved its metrics.
        if metrics_path.exists():                # Skip runs that have not been done yet.
            rows.append(json.loads(metrics_path.read_text()))  # Read that run's metrics.
names = [f"{'BatchNorm' if m['batch_norm'] else 'Original'} CNN\n{'with' if m['augmentation'] else 'no'} augmentation"
         for m in rows]                          # A short label for each run.

print(f"{'Network':<10}{'Augment':<9}{'Precision':>11}{'Precision':>11}{'Accuracy':>10}{'Balanced':>10}{'Best':>6}")  # Table header, line 1.
print(f"{'':<10}{'':<9}{'(macro)':>11}{'(weighted)':>11}{'':>10}{'accuracy':>10}{'epoch':>6}")                    # Table header, line 2.
for m in rows:                                   # One table row per run.
    print(f"{'BatchNorm' if m['batch_norm'] else 'Original':<10}{'yes' if m['augmentation'] else 'no':<9}"
          f"{m['precision_macro']:>11.2%}{m['precision_weighted']:>11.2%}"
          f"{m['test_accuracy']:>10.2%}{m['balanced_accuracy']:>10.2%}{m['best_epoch']:>6}")

print('\nPrecision per class:')                  # Second table: precision of each flower class.
print(f"{'Run':<24}" + ''.join(f'{name:>12}' for name in class_names))  # Header with class names.
for name, m in zip(names, rows):
    print(f"{name.replace(chr(10), ', '):<24}" + ''.join(f"{m['precision_per_class'][c]:>12.2%}" for c in class_names))

fig, ax = plt.subplots(figsize=(9, 4))           # Bar chart of overall (macro) precision.
bars = ax.bar(names, [m['precision_macro'] for m in rows], color=['#9db4d3', '#4a78b5', '#e0a96d', '#c4661f'][:len(rows)])  # One bar per run.
ax.bar_label(bars, labels=[f"{m['precision_macro']:.1%}" for m in rows])  # Write the value on each bar.
ax.set(ylabel='Overall precision (macro)', ylim=(0, 1),
       title='Overall precision on the full test set (551 images)')
ax.grid(axis='y', alpha=0.25)                    # Light horizontal grid.
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Validation curves of all runs, side by side.
for m, name in zip(rows, names):                 # One line per run.
    axes[0].plot(range(1, len(m['history']['val_loss']) + 1), m['history']['val_loss'], label=name.replace('\n', ', '))
    axes[1].plot(range(1, len(m['history']['val_accuracy']) + 1), m['history']['val_accuracy'], label=name.replace('\n', ', '))
axes[0].set(xlabel='Epoch', ylabel='Validation loss', title='Validation loss by epoch')
axes[1].set(xlabel='Epoch', ylabel='Validation accuracy', title='Validation accuracy by epoch', ylim=(0, 1))
for ax in axes:
    ax.legend(fontsize=8)                        # Legend for each plot.
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 12. Report on results

_Fill in this section after the four runs, using the tables and charts in section 11._

| Network | Augmentation | Overall precision (macro) | Overall precision (weighted) | Test accuracy | Best epoch |
|---|---|---|---|---|---|
| Original `BasicCNN` | No | | | | |
| Original `BasicCNN` | Yes | | | | |
| `BatchNormCNN` | No | | | | |
| `BatchNormCNN` | Yes | | | | |

Points to discuss:

1. **Effect of Batch Normalization without augmentation:** compare rows 1 and 3. Did precision go up or down, and by how much?
2. **Effect of Batch Normalization with augmentation:** compare rows 2 and 4.
3. **Effect of augmentation on each network:** compare rows 1 vs 2 and rows 3 vs 4.
4. **Training behaviour:** in the validation curves, did BatchNorm reach a low validation loss in fewer epochs (see the best epoch column)? Did either network overfit (validation loss rising while training loss keeps falling)?
5. **Per-class precision:** which flower classes improved or got worse (for example roses vs tulips, which are often confused)?
6. **Caveat:** each number comes from one run with one seed on 551 test images, so differences of 1–2 percentage points may be noise rather than a real effect.